# Celeb-DF v2 Cross-Dataset Generalization Evaluation
### Evaluating Final EFFResNet-Unified Architecture on Unseen Deepfakes

**Evaluation Protocol:**
1. **Model**: Unified architecture trained on FaceForensics++ (`best_effresnet_unified.pth`).
2. **Dataset**: Celeb-DF v2 (Unseen subjects, unseen deepfake generation algorithms, unseen compression).
3. **Metrics**: Single-Frame Accuracy & AUC-ROC vs. Multi-Frame Temporal Video Voting.
4. **Scientific Goal**: Demonstrate cross-dataset generalizability and zero-shot domain transfer.

In [ ]:
import os, sys, time, json, random
from pathlib import Path
from typing import List, Tuple, Dict
from collections import defaultdict

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.models as models

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, roc_curve
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('=' * 65)
print('PyTorch Version :', torch.__version__)
print('Device          :', device)
if torch.cuda.is_available():
    print('GPU Device      :', torch.cuda.get_device_name(0))
    print('VRAM Available  :', round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2), 'GB')
print('=' * 65)

In [ ]:
# ── CHECKPOINT DISCOVERY ─────────────────────────────────────────
def find_checkpoint() -> Path:
    for p in Path('/kaggle/input').rglob('best_effresnet_unified.pth'):
        if p.is_file(): return p
    for p in Path('.').rglob('best_effresnet_unified.pth'):
        if p.is_file(): return p
    raise FileNotFoundError('Could not locate best_effresnet_unified.pth under /kaggle/input')

CHECKPOINT_PATH = find_checkpoint()
print('Located Trained Unified Checkpoint at:', CHECKPOINT_PATH)
print(f'File Size: {CHECKPOINT_PATH.stat().st_size / (1024**2):.2f} MB')

In [ ]:
# ── CELEB-DF V2 DATASET DISCOVERY & MANIFEST ────────────────────
def locate_celebdf_root() -> Tuple[Path, Path]:
    # Search for directories containing 'real' and 'fake'
    for r in Path('/kaggle/input').rglob('real'):
        if r.is_dir():
            parent = r.parent
            f = parent / 'fake'
            if f.exists() and f.is_dir():
                # Make sure it's Celeb-DF, not FF++
                if 'celeb' in str(parent).lower() or 'celeb_v2' in str(parent).lower():
                    return r, f
    for r in Path('/kaggle/input').rglob('real'):
        if r.is_dir() and (r.parent / 'fake').exists():
            return r, r.parent / 'fake'
    raise FileNotFoundError('Celeb-DF dataset not found under /kaggle/input')

real_dir, fake_dir = locate_celebdf_root()
print('Discovered Real Faces Folder:', real_dir)
print('Discovered Fake Faces Folder:', fake_dir)

def extract_video_id(filename: str) -> str:
    stem = Path(filename).stem
    if '_face_' in stem: return stem.split('_face_')[0]
    if '_frame' in stem: return stem.split('_frame')[0]
    if '_face' in stem:  return stem.split('_face')[0]
    parts = stem.split('_')
    return '_'.join(parts[:3]) if len(parts) >= 3 else stem

def build_celebdf_manifest(real_dir: Path, fake_dir: Path, max_per_video: int = 15, seed: int = 42):
    random.seed(seed)
    real_vids = defaultdict(list)
    for p in sorted(real_dir.rglob('*.*')):
        if p.suffix.lower() in ['.jpg', '.png', '.jpeg']:
            vid = extract_video_id(p.name)
            real_vids[vid].append(str(p))

    fake_vids = defaultdict(list)
    for p in sorted(fake_dir.rglob('*.*')):
        if p.suffix.lower() in ['.jpg', '.png', '.jpeg']:
            vid = extract_video_id(p.name)
            fake_vids[vid].append(str(p))

    print(f'Unique Real Videos in Celeb-DF : {len(real_vids)}')
    print(f'Unique Fake Videos in Celeb-DF : {len(fake_vids)}')

    def sample_entries(vid_dict, label):
        samples = []
        for v, fl in sorted(vid_dict.items()):
            fl = sorted(fl)
            chosen = fl if len(fl) <= max_per_video else [
                fl[int(i * len(fl) / max_per_video)] for i in range(max_per_video)
            ]
            for f in chosen:
                samples.append((f, label, v))
        return samples

    real_samples = sample_entries(real_vids, 0)
    fake_samples = sample_entries(fake_vids, 1)

    # Balanced evaluation across both classes
    n_eval = min(len(real_samples), len(fake_samples))
    eval_samples = random.sample(real_samples, n_eval) + random.sample(fake_samples, n_eval)
    random.shuffle(eval_samples)
    return eval_samples

test_entries = build_celebdf_manifest(real_dir, fake_dir, max_per_video=15)
unique_test_vids = len(set(e[2] for e in test_entries))
print(f'Total Evaluated Celeb-DF Frames : {len(test_entries):,} (Balanced Real/Fake)')
print(f'Total Unique Evaluated Videos   : {unique_test_vids:,}')

In [ ]:
# ── DATASET & DATALOADER ─────────────────────────────────────────
class CelebDFEvalDataset(Dataset):
    def __init__(self, entries, transform=None):
        self.entries = entries
        self.transform = transform
    def __len__(self): return len(self.entries)
    def __getitem__(self, idx):
        path, label, vid_id = self.entries[idx]
        try:    img = Image.open(path).convert('RGB')
        except: img = Image.new('RGB', (224, 224))
        if self.transform: img = self.transform(img)
        else: img = transforms.ToTensor()(img)
        return img, torch.tensor(label, dtype=torch.float32), vid_id

eval_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

eval_loader = DataLoader(CelebDFEvalDataset(test_entries, eval_tf), batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
print(f'Evaluation Batches: {len(eval_loader)}')

In [ ]:
# ── MODEL ARCHITECTURE & WEIGHT LOADING ──────────────────────────
def get_srm_filters() -> torch.Tensor:
    f1 = np.array([
        [0,  0,  0,  0,  0],
        [0, -1,  2, -1,  0],
        [0,  2, -4,  2,  0],
        [0, -1,  2, -1,  0],
        [0,  0,  0,  0,  0]
    ], dtype=np.float32) / 4.0
    f2 = np.array([
        [-1,  2, -2,  2, -1],
        [ 2, -6,  8, -6,  2],
        [-2,  8,-12,  8, -2],
        [ 2, -6,  8, -6,  2],
        [-1,  2, -2,  2, -1]
    ], dtype=np.float32) / 12.0
    f3 = np.array([
        [0,  0,  0,  0,  0],
        [0,  1, -2,  1,  0],
        [0, -2,  4, -2,  0],
        [0,  1, -2,  1,  0],
        [0,  0,  0,  0,  0]
    ], dtype=np.float32) / 4.0
    filters = np.stack([f1, f2, f3], axis=0)
    return torch.tensor(np.expand_dims(filters, axis=1), dtype=torch.float32)

class SRMSteganalysisStream(nn.Module):
    def __init__(self, out_dim=256):
        super().__init__()
        srm_k = get_srm_filters()
        self.srm_conv = nn.Conv2d(3, 3, kernel_size=5, padding=2, groups=3, bias=False)
        self.srm_conv.weight.data = srm_k.repeat(1, 1, 1, 1)
        self.srm_conv.weight.requires_grad = False
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride=2, padding=1, bias=False), nn.BatchNorm2d(32), nn.GELU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1, bias=False), nn.BatchNorm2d(64), nn.GELU(),
            nn.Conv2d(64, 128, 3, stride=2, padding=1, bias=False), nn.BatchNorm2d(128), nn.GELU(),
            nn.Conv2d(128, 256, 3, stride=2, padding=1, bias=False), nn.BatchNorm2d(256), nn.GELU(),
            nn.Conv2d(256, out_dim, 3, stride=2, padding=1, bias=False), nn.BatchNorm2d(out_dim), nn.GELU()
        )
    def forward(self, x): return self.encoder(self.srm_conv(x))

class SpectralFrequencyStream(nn.Module):
    def __init__(self, out_dim=256):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride=2, padding=1, bias=False), nn.BatchNorm2d(32), nn.GELU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1, bias=False), nn.BatchNorm2d(64), nn.GELU(),
            nn.Conv2d(64, 128, 3, stride=2, padding=1, bias=False), nn.BatchNorm2d(128), nn.GELU(),
            nn.Conv2d(128, 256, 3, stride=2, padding=1, bias=False), nn.BatchNorm2d(256), nn.GELU(),
            nn.Conv2d(256, out_dim, 3, stride=2, padding=1, bias=False), nn.BatchNorm2d(out_dim), nn.GELU()
        )
    def forward(self, x):
        fft2d = torch.fft.fft2(x, norm='ortho')
        mag = torch.log(torch.abs(fft2d) + 1e-6)
        mag_shift = torch.fft.fftshift(mag, dim=(-2, -1))
        return self.encoder(mag_shift)

class ConvLocalityBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.act1 = nn.GELU()
        self.depthwise = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, groups=out_channels, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.act2 = nn.GELU()
    def forward(self, x): return self.act2(self.bn2(self.depthwise(self.act1(self.bn1(self.conv1(x))))))

class CrossAttentionBlock(nn.Module):
    def __init__(self, embed_dim, num_heads=8, dropout=0.1):
        super().__init__()
        self.norm_q = nn.LayerNorm(embed_dim)
        self.norm_kv = nn.LayerNorm(embed_dim)
        self.mha = nn.MultiheadAttention(embed_dim=embed_dim, num_heads=num_heads, dropout=dropout, batch_first=True)
        self.dropout = nn.Dropout(dropout)
    def forward(self, q, kv):
        q_n = self.norm_q(q)
        kv_n = self.norm_kv(kv)
        out, _ = self.mha(q_n, kv_n, kv_n)
        return self.dropout(out)

class TransformerEncoderBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, mlp_ratio=4.0, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.mha = nn.MultiheadAttention(embed_dim=embed_dim, num_heads=num_heads, dropout=dropout, batch_first=True)
        self.dropout1 = nn.Dropout(dropout)
        self.norm2 = nn.LayerNorm(embed_dim)
        mlp_hidden_dim = int(embed_dim * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, mlp_hidden_dim), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(mlp_hidden_dim, embed_dim), nn.Dropout(dropout)
        )
    def forward(self, x):
        normed = self.norm1(x)
        attn_out, _ = self.mha(normed, normed, normed)
        x = x + self.dropout1(attn_out)
        x = x + self.mlp(self.norm2(x))
        return x

class EFFResNetUnified(nn.Module):
    def __init__(self, embed_dim=256, num_heads=8, num_vit_layers=4, mlp_ratio=4.0, dropout=0.1):
        super().__init__()
        self.embed_dim = embed_dim
        eff = models.efficientnet_b0(weights=None)
        self.eff_features = eff.features
        res = models.resnet50(weights=None)
        self.res_stem = nn.Sequential(
            res.conv1, res.bn1, res.relu, res.maxpool,
            res.layer1, res.layer2, res.layer3, res.layer4
        )
        self.fusion_neck = ConvLocalityBlock(1280 + 2048, embed_dim)
        self.freq_stream = SpectralFrequencyStream(embed_dim)
        self.srm_stream = SRMSteganalysisStream(embed_dim)
        self.cross_attn_freq = CrossAttentionBlock(embed_dim, num_heads=num_heads, dropout=dropout)
        self.cross_attn_srm = CrossAttentionBlock(embed_dim, num_heads=num_heads, dropout=dropout)
        self.alpha_freq = nn.Parameter(torch.zeros(1))
        self.alpha_srm = nn.Parameter(torch.zeros(1))
        self.num_patches = 7 * 7
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, self.num_patches + 1, embed_dim))
        self.pos_dropout = nn.Dropout(dropout)
        self.transformer_layers = nn.ModuleList([
            TransformerEncoderBlock(embed_dim, num_heads, mlp_ratio, dropout)
            for _ in range(num_vit_layers)
        ])
        self.norm = nn.LayerNorm(embed_dim)
        self.post_attn_locality = nn.Sequential(
            nn.Conv2d(embed_dim, embed_dim, kernel_size=3, padding=1, groups=embed_dim, bias=False),
            nn.BatchNorm2d(embed_dim),
            nn.GELU()
        )
        self.feat_proj = nn.Linear(embed_dim * 2, embed_dim)
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim, 128), nn.GELU(), nn.Dropout(dropout), nn.Linear(128, 1)
        )
    def forward(self, x):
        B = x.size(0)
        f_eff = self.eff_features(x)
        f_res = self.res_stem(x)
        if f_eff.shape[2:] != f_res.shape[2:]:
            f_eff = F.interpolate(f_eff, size=f_res.shape[2:], mode='bilinear', align_corners=False)
        f_spatial = self.fusion_neck(torch.cat([f_eff, f_res], dim=1))
        f_freq = self.freq_stream(x)
        f_srm = self.srm_stream(x)
        H, W = f_spatial.shape[2], f_spatial.shape[3]
        tokens_spatial = f_spatial.flatten(2).transpose(1, 2)
        tokens_freq = f_freq.flatten(2).transpose(1, 2)
        tokens_srm = f_srm.flatten(2).transpose(1, 2)
        freq_context = self.cross_attn_freq(tokens_spatial, tokens_freq)
        srm_context = self.cross_attn_srm(tokens_spatial, tokens_srm)
        tokens_fused = (
            tokens_spatial
            + torch.tanh(self.alpha_freq) * freq_context
            + torch.tanh(self.alpha_srm) * srm_context
        )
        cls_tokens = self.cls_token.expand(B, -1, -1)
        tokens = torch.cat([cls_tokens, tokens_fused], dim=1)
        tokens = self.pos_dropout(tokens + self.pos_embed[:, :tokens.size(1), :])
        for layer in self.transformer_layers: tokens = layer(tokens)
        tokens = self.norm(tokens)
        cls_out = tokens[:, 0]
        spatial_tokens = tokens[:, 1:]
        spatial_grid = spatial_tokens.transpose(1, 2).reshape(B, self.embed_dim, H, W)
        spatial_recovered = self.post_attn_locality(spatial_grid)
        spatial_pool = F.adaptive_avg_pool2d(spatial_recovered, (1, 1)).flatten(1)
        feat = self.feat_proj(torch.cat([cls_out, spatial_pool], dim=1))
        return self.classifier(feat)

model = EFFResNetUnified(embed_dim=256, num_heads=8, num_vit_layers=4).to(device)
try:
    ckpt = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)
except TypeError:
    ckpt = torch.load(CHECKPOINT_PATH, map_location=device)
if 'model_state_dict' in ckpt: model.load_state_dict(ckpt['model_state_dict'])
else: model.load_state_dict(ckpt)
model.eval()
print('Successfully loaded trained EFFResNet-Unified model!')
print(f'Gated Parameters: tanh(alpha_freq)={torch.tanh(model.alpha_freq).item():.5f}, tanh(alpha_srm)={torch.tanh(model.alpha_srm).item():.5f}')

In [ ]:
# ── CROSS-DATASET INFERENCE EXECUTION ───────────────────────────
all_probs = []
all_labels = []
all_vids = []

t0 = time.time()
print('Running inference on unseen Celeb-DF v2 dataset...')
with torch.no_grad():
    for step, (imgs, lbls, vids) in enumerate(eval_loader):
        imgs = imgs.to(device, non_blocking=True)
        with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
            logits = model(imgs)
            probs = torch.sigmoid(logits)
        all_probs.extend(probs.squeeze(1).cpu().numpy().tolist())
        all_labels.extend(lbls.numpy().tolist())
        all_vids.extend(vids)
        if (step + 1) % 25 == 0 or (step + 1) == len(eval_loader):
            print(f'Processed [{step+1}/{len(eval_loader)}] batches...')

elapsed = time.time() - t0
print(f'Inference finished in {elapsed:.1f}s ({len(all_probs)/elapsed:.1f} frames/sec)')

In [ ]:
# ── METRICS COMPUTATION: FRAME VS VIDEO AGGREGATION ─────────────
def compute_metrics(y_true, y_prob, thr=0.5):
    y_pred = (y_prob >= thr).astype(int)
    acc = accuracy_score(y_true, y_pred) * 100.0
    prec = precision_score(y_true, y_pred, zero_division=0) * 100.0
    rec = recall_score(y_true, y_pred, zero_division=0) * 100.0
    f1 = f1_score(y_true, y_pred, zero_division=0) * 100.0
    auc = roc_auc_score(y_true, y_prob) * 100.0 if len(set(y_true)) > 1 else 50.0
    cm = confusion_matrix(y_true, y_pred).tolist()
    return {'accuracy': round(acc, 2), 'precision': round(prec, 2), 'recall': round(rec, 2), 'f1_score': round(f1, 2), 'auc_roc': round(auc, 2), 'confusion_matrix': cm}

# 1. Frame-Level Metrics
frame_metrics = compute_metrics(np.array(all_labels), np.array(all_probs))

# 2. Video-Level Voting Metrics
vid_probs = defaultdict(list)
vid_labels = {}
for p, y, vid in zip(all_probs, all_labels, all_vids):
    vid_probs[vid].append(p)
    vid_labels[vid] = y

sorted_vids = sorted(vid_probs.keys())
v_true = np.array([vid_labels[v] for v in sorted_vids])

# Mean Soft Voting
v_mean_prob = np.array([np.mean(vid_probs[v]) for v in sorted_vids])
video_mean_metrics = compute_metrics(v_true, v_mean_prob)

# Median Robust Voting
v_median_prob = np.array([np.median(vid_probs[v]) for v in sorted_vids])
video_median_metrics = compute_metrics(v_true, v_median_prob)

# Majority Hard Voting
v_maj_pred = np.array([int(np.mean([p >= 0.5 for p in vid_probs[v]]) >= 0.5) for v in sorted_vids])
video_maj_acc = round(accuracy_score(v_true, v_maj_pred) * 100.0, 2)

print('=' * 85)
print('CROSS-DATASET GENERALIZATION RESULTS ON CELEB-DF V2')
print('=' * 85)
print(f'1. Single-Frame Evaluation  : Accuracy = {frame_metrics["accuracy"]}% | AUC = {frame_metrics["auc_roc"]}% | Recall = {frame_metrics["recall"]}%')
print(f'2. Video Mean Voting        : Accuracy = {video_mean_metrics["accuracy"]}% | AUC = {video_mean_metrics["auc_roc"]}% | Precision = {video_mean_metrics["precision"]}%')
print(f'3. Video Median Voting      : Accuracy = {video_median_metrics["accuracy"]}% | AUC = {video_median_metrics["auc_roc"]}%')
print(f'4. Video Majority Hard Vote : Accuracy = {video_maj_acc}%')
print('=' * 85)

OUTPUT_DIR = Path('/kaggle/working/output_celebdf_eval')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
summary = {
    'frame_level': frame_metrics,
    'video_mean_voting': video_mean_metrics,
    'video_median_voting': video_median_metrics,
    'video_majority_acc': video_maj_acc,
    'total_frames': len(all_labels),
    'total_videos': len(sorted_vids)
}
with open(OUTPUT_DIR / 'celebdf_eval_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

# Plot Video Confusion Matrix
cm = np.array(video_mean_metrics['confusion_matrix'])
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Real', 'Fake'], yticklabels=['Real', 'Fake'])
plt.title(f'Celeb-DF v2 Cross-Dataset Confusion Matrix (Acc: {video_mean_metrics["accuracy"]}%)')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'celebdf_confusion_matrix.png', dpi=300)
plt.show()
print('Saved results to', OUTPUT_DIR)